# Basic Local Aignment Search Tool

This notebook gives some quick info to the BLASTing in the project. All the direct work on BLAST takes place outside of this project folder, as it is problematic connecting OneDrive with BLAST. In my experience. The work was done with Git Bash.\
The reference genome was downloaded as such:

Further on, from this, the database was created, as one needs to have something to BLAST against. The database is included in ecoli_db. Just for show...\
Then, the BLAST could proceed. The file obtained for transporters is named approac1.fasta. The limits can also easily be adjusted. Here, it is shown with E-value < 1e-5. -outfmt is the tabular output.

The output looks like this:

Where the columns respectively are:\
Query_ID  Subject_ID  Identity  Alignment_Length  Mismatches  Gap_Openings  Query_Start  Query_End  Subject_Start  Subject_End  E-value  Bit-Score

Next up, the results were obtained, and could also be filtered to only retain the highest scoring sequences. This filter keeps only the instances where the Identity > 40% matched, and E-value < 1e-5. The E-val part is in this example useless, as it was already applied in the BLASTing section, but can easily be tuned.

Respectively, all the matches can easily be counted, and all the IDs from approach1.fasta can also be listed as such:

The file is ready for use and further processing now. Well, it was before these last lines as well. But they just give a nice and quick overview.

Below follows an example of how the file can be processed to obtain the desired info on the transport reactions of E. coli.

In [1]:
import pandas as pd


In [31]:
blast_results = pd.read_csv("results_a1.txt", sep="\t", header=None)

blast_results.columns = [
    "QueryID", "SubjectID", "Identity", "AlignLength", "Mismatches", 
    "GapOpens", "QStart", "QEnd", "SStart", "SEnd", "E-value", "BitScore"
]

eval_threshold = 1e-5
id_threshold = 30

filtered_hits = blast_results[
    (blast_results["E-value"] <= eval_threshold) & 
    (blast_results["Identity"] >= id_threshold)
]

matched_ids = set(filtered_hits["QueryID"].tolist())
matched_ids
filtered_hits[["UID", "TCID"]] = filtered_hits["QueryID"].str.split("|", expand=True)
filtered_hits = filtered_hits.drop(columns=['QueryID'])
cols = ['UID', 'TCID'] + [col for col in filtered_hits.columns if col not in ['UID', 'TCID']]
filtered_hits = filtered_hits[cols]
filtered_hits

C:\Users\landr\AppData\Local\Temp\ipykernel_8420\1433853509.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_hits[["UID", "TCID"]] = filtered_hits["QueryID"].str.split("|", expand=True)
C:\Users\landr\AppData\Local\Temp\ipykernel_8420\1433853509.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_hits[["UID", "TCID"]] = filtered_hits["QueryID"].str.split("|", expand=True)


,UID,TCID,SubjectID,Identity,AlignLength,Mismatches,GapOpens,QStart,QEnd,SStart,SEnd,E-value,BitScore
1,A0JCJ5,1.B.1.1.7,sp|P76335|YEDS_ECOLI,38.686,137,78,4,1,134,1,134,4.720000e-16,72.0
3,A0JCJ5,1.B.1.1.7,sp|P06996|OMPC_ECOLI,35.878,131,78,2,1,131,1,125,1.220000e-13,68.2
6,A0JCJ5,1.B.1.1.7,sp|P77519|YDDL_ECOLI,36.264,91,54,2,1,89,1,89,3.180000e-08,48.1
8,A0L4L0,3.A.1.132.3,sp|P0AF08|APBC_ECOLI,39.377,353,175,8,2,329,12,350,4.320000e-70,220.0
9,A0L4L0,3.A.1.132.3,sp|P0AEZ3|MIND_ECOLI,43.396,53,30,0,91,143,3,55,1.320000e-06,46.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
118081,VZT40043.1,4.A.2.1.28,sp|P0AA04|PTHP_ECOLI,31.579,76,52,0,9,84,6,81,8.020000e-07,40.4
118083,MFA7902620.1,4.A.2.1.28,sp|P69829|PTSN_ECOLI,47.020,151,78,2,2,151,8,157,3.830000e-37,122.0
118086,MFA7902620.1,4.A.2.1.28,sp|P69811|PTFAH_ECOLI,30.075,133,78,5,18,143,11,135,6.760000e-08,47.4
118090,WRI60233.1,3.D.4.11.2,sp|P0ABI8|CYOB_ECOLI,36.242,447,275,5,11,451,46,488,5.350000e-93,294.0


Now, merging based on the data BLASTed againts.

In [34]:
a1 = pd.read_csv("../Approach 1/a1_df.tsv", sep="\t")
results = pd.merge(a1, filtered_hits, on=["UID", "TCID"], how="inner")
results


,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,...,Identity,AlignLength,Mismatches,GapOpens,QStart,QEnd,SStart,SEnd,E-value,BitScore
0,A0JCJ5,1.B.1.1.7,CHEBI:25367,molecule,MKKTILALAVPALLAAGVTNAATVYNNDGTKIDLKGSIRLLAEDGA...,1.B.1,Solute (out) ⇌ Solute (in),Solute,molecule (out) ⇌ molecule (in),NaN,...,38.686,137,78,4,1,134,1,134,4.720000e-16,72.0
1,A0JCJ5,1.B.1.1.7,CHEBI:25367,molecule,MKKTILALAVPALLAAGVTNAATVYNNDGTKIDLKGSIRLLAEDGA...,1.B.1,Solute (out) ⇌ Solute (in),Solute,molecule (out) ⇌ molecule (in),NaN,...,35.878,131,78,2,1,131,1,125,1.220000e-13,68.2
2,A0JCJ5,1.B.1.1.7,CHEBI:25367,molecule,MKKTILALAVPALLAAGVTNAATVYNNDGTKIDLKGSIRLLAEDGA...,1.B.1,Solute (out) ⇌ Solute (in),Solute,molecule (out) ⇌ molecule (in),NaN,...,36.264,91,54,2,1,89,1,89,3.180000e-08,48.1
3,A0L4L0,3.A.1.132.3,NaN,NaN,MAQRAAIVALFDQLQEPKLKWNINTLNLLQEVTLHEQHLRVVVHLI...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,Solute (in) + ATP → Solute (out) + ADP + Pi,NaN,...,39.377,353,175,8,2,329,12,350,4.320000e-70,220.0
4,A0L4L0,3.A.1.132.3,NaN,NaN,MAQRAAIVALFDQLQEPKLKWNINTLNLLQEVTLHEQHLRVVVHLI...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,Solute (in) + ATP → Solute (out) + ADP + Pi,NaN,...,43.396,53,30,0,91,143,3,55,1.320000e-06,46.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157813,MFA7902620.1,4.A.2.1.28,CHEBI:14911,protein,MIRLETILTPGRSLVNVPGGSKKRALEKVATLIADQVPELEMQDVF...,4.A.2,NaN,NaN,NaN,NaN,...,47.020,151,78,2,2,151,8,157,3.830000e-37,122.0
157814,MFA7902620.1,4.A.2.1.28,CHEBI:14911,protein,MIRLETILTPGRSLVNVPGGSKKRALEKVATLIADQVPELEMQDVF...,4.A.2,NaN,NaN,NaN,NaN,...,30.075,133,78,5,18,143,11,135,6.760000e-08,47.4
157815,WRI60233.1,3.D.4.11.2,CHEBI:15378,hydron,MLNFKYYSGISNWLESSNHKDIGTLYFIFGLWSGMLGTSLSMIIRF...,3.D.4,NaN,NaN,NaN,NaN,...,36.242,447,275,5,11,451,46,488,5.350000e-93,294.0
157816,WRI60233.1,3.D.4.11.2,CHEBI:10545,electron,MLNFKYYSGISNWLESSNHKDIGTLYFIFGLWSGMLGTSLSMIIRF...,3.D.4,NaN,NaN,NaN,NaN,...,36.242,447,275,5,11,451,46,488,5.350000e-93,294.0


Alright, this concludes the BLAST-section. The columns and what to keep, remain for the pipeline to take control of. It is better to keep as much info as possible until the final pipeline is done. Then, optimization might be correct to put in place. Above is described a way to solve the issue on merging BLAST results against the transporters from TCDB. Next step is to actually build the pipeline!